Formataçãodo do dados do DataFrame de Infodengue 

In [9]:
import pandas as pd

# Base escolhida: InfoDengue
# Cidade analisada: Fortaleza
# Período da consulta: 2017 até 2024

geocode = 2304400
doenca = "dengue"
ano_inicial = 2010
ano_final = 2024
semana_inicial = 1
semana_final = 53

arquivo_csv = "dados_infodengue/infodengue_fortaleza_2017_2024.csv"
arquivo_relatorio = "dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV"

# montando a url da API
url = (
    "https://info.dengue.mat.br/api/alertcity?"
    f"geocode={geocode}"
    f"&disease={doenca}"
    f"&format=csv"
    f"&ew_start={semana_inicial}"
    f"&ew_end={semana_final}"
    f"&ey_start={ano_inicial}"
    f"&ey_end={ano_final}"
)

print("URL usada:")
print(url)

# leitura da base
df = pd.read_csv(url)

print("\nBase carregada com sucesso")
print("Dimensão da base:", df.shape)

print("\nPrimeiras linhas:")
print(df.head())

print("\nColunas encontradas:")
print(df.columns.tolist())

# convertendo a coluna de data
df["data_iniSE"] = pd.to_datetime(df["data_iniSE"], errors="coerce")

# salvando o csv baixado
df.to_csv(arquivo_csv, index=False, encoding="utf-8")
print(f"\nCSV salvo com sucesso: {arquivo_csv}")

URL usada:
https://info.dengue.mat.br/api/alertcity?geocode=2304400&disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2024

Base carregada com sucesso
Dimensão da base: (782, 31)

Primeiras linhas:
   data_iniSE      SE  casos_est  casos_est_min  casos_est_max  casos  \
0  2024-12-22  202452       42.0             42             42     42   
1  2024-12-15  202451       65.0             65             65     65   
2  2024-12-08  202450       82.0             82             82     82   
3  2024-12-01  202449       85.0             85             85     85   
4  2024-11-24  202448       99.0             99             99     99   

      p_rt1  p_inc100k  Localidade_id  nivel  ...    umidmed    umidmin  \
0  0.000549   1.617776              0      1  ...  75.053529  57.029057   
1  0.027220   2.503701              0      1  ...  78.271629  61.219586   
2  0.075172   3.158515              0      1  ...  75.892571  59.095457   
3  0.039801   3.274070              0      1 

Início do processo de validação dos dados, selecionando as colunas relevantes para análise

In [10]:
#Pegando as colunas de interesse para o relatório
colunas_interesse = ["data_iniSE", "casos_est"]

# Gerando dataframe com as colunas de interesse
df_relatorio = df[colunas_interesse]

#alterando o nome das colunas para o relatório
df_relatorio.rename(columns={"data_iniSE": "data", "casos_est": "casos_estimados"}, inplace=True)
    
print(df_relatorio.head())

        data  casos_estimados
0 2024-12-22             42.0
1 2024-12-15             65.0
2 2024-12-08             82.0
3 2024-12-01             85.0
4 2024-11-24             99.0


Soma de caso obtidos por mês 

In [11]:
#Agrupando por mês e somando os casos estimados
#O 'MS' significa Month Start (Início do Mês)
df_mensal = df_relatorio.set_index("data").resample("MS")['casos_estimados'].sum().reset_index()

#renomeando a coluna de casos estimados para casos_mensais e data para DATA_LOCAL
df_mensal.rename(columns={"casos_estimados": "casos_mensais", "data": "MES_REFERENCIA"}, inplace=True)

#readicionando a coluna de nível de alerta, pegando o valor mais frequente do nível de alerta para cada mês


print(df_mensal.head())

  MES_REFERENCIA  casos_mensais
0     2010-01-01          388.0
1     2010-02-01          301.0
2     2010-03-01          337.0
3     2010-04-01          388.0
4     2010-05-01          514.0


Adicionando uma coluna de nivel de alerta

In [15]:
import numpy as np

#Calcular a média e o desvio padrão histórico de fortaleza
media_casos  = df_mensal["casos_mensais"].mean()
desvio_padrao = df_mensal["casos_mensais"].std()

#Defenir a lógica de alerta
def definir_alerta(casos):
    if casos <= media_casos:
        return 1 # Alerta baixo
    elif casos <= (media_casos + desvio_padrao):
        return 2 # Alerta médio
    elif casos <= (media_casos + 2 * desvio_padrao):
        return 3 # Alerta alto
    else:
        return 4 # Alerta crítico
    

# Criando a nova coluna de nível de alerta
df_mensal["nivel_alerta"] = df_mensal['casos_mensais'].apply(definir_alerta) 

print(df_mensal.head())

  MES_REFERENCIA  casos_mensais  nivel_alerta
0     2010-01-01          388.0             1
1     2010-02-01          301.0             1
2     2010-03-01          337.0             1
3     2010-04-01          388.0             1
4     2010-05-01          514.0             1


Exporta o Dataframe

In [13]:
#exportando o dataframe para um arquivo csv
df_mensal.to_csv(arquivo_relatorio, index=False, encoding="utf-8")
print(f"\nRelatório salvo com sucesso: {arquivo_relatorio}")


Relatório salvo com sucesso: dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV
